# SemIf · Evaluation

<a href="https://colab.research.google.com/github/furbyhaxx/semlf/blob/main/notebooks/SemIf_Eval.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Measure decision quality, speed and memory for any model and precision on the owned benchmark suites, and check how far the results drift from the committed BF16 Qwen3.5-4B reference. Use it to find the cheapest setting for an older or smaller GPU that keeps accuracy, or to screen other base models.

To run this, press *Runtime* → *Run all* on a **free** Tesla T4 Colab instance.
Companion notebooks: [inference](https://colab.research.google.com/github/furbyhaxx/semlf/blob/main/notebooks/SemIf_Inference.ipynb) · [evaluation](https://colab.research.google.com/github/furbyhaxx/semlf/blob/main/notebooks/SemIf_Eval.ipynb) · [training](https://colab.research.google.com/github/furbyhaxx/semlf/blob/main/notebooks/SemIf_Train.ipynb).

### Installation

Same stack as Unsloth's Qwen3.5 (4B) notebook: Unsloth, its bundled gated-delta-net
kernels and `causal_conv1d` for the Qwen3.5 linear-attention layers. No vLLM:
every decision is one forward pass, so there is nothing to serve.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# Unsloth bundles the gated delta net kernels; a leftover pip fla would shadow them
!uv pip uninstall -qqq flash-linear-attention fla-core
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation causal_conv1d==1.6.0
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# @title Fetch SemIf { display-mode: "form" }
# SemIf is put on sys.path instead of pip-installed: its pyproject pins torch 2.10,
# which would replace the torch Unsloth was installed against.
import json, subprocess, sys, time
from pathlib import Path

SEMLF_REPO = "https://github.com/furbyhaxx/semlf"  # @param {type:"string"}
SEMLF_REF = "main"  # @param {type:"string"}

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "semif_phase1").is_dir()), None)
if ROOT is None:
    ROOT = (Path("/content") if Path("/content").is_dir() else Path.cwd()) / "semlf"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", SEMLF_REF, SEMLF_REPO, str(ROOT)], check=True)
for path in (ROOT / "src", ROOT / "notebooks"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
COMMIT = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"SemIf at {ROOT} ({COMMIT[:12] or 'not a git checkout'}); outputs go to {OUT}")

### Configuration

Pick a pinned model from the list or type any Hub ID, local checkpoint, or a LoRA adapter directory saved by the training notebook.

In [ ]:
# @title Model { display-mode: "form" }
MODELS = {  # pinned revisions; any other Hub model or local path also works
    "Qwen/Qwen3.5-4B": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",  # reference: committed BF16 predictions
    "Qwen/Qwen3.5-2B": "15852e8c16360a2fea060d615a32b45270f8a8fc",
    "Qwen/Qwen3.5-0.8B": "2fc06364715b967f1860aea9cf38778875588b17",
    "Qwen/Qwen3.5-9B": "c202236235762e1c871ad0ccb60c8ee5ba337b9a",
    "Qwen/Qwen3-4B-Instruct-2507": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "Qwen/Qwen3-1.7B": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
    "Qwen/Qwen3-0.6B": "c1899de289a04d12100db370d81485cdf75e47ca",
    "openbmb/MiniCPM5-2B": "12a3808a956f869c767195e9266b59c4d21d92e2",
}

MODEL = "Qwen/Qwen3.5-4B"  # @param ["Qwen/Qwen3.5-4B", "Qwen/Qwen3.5-2B", "Qwen/Qwen3.5-0.8B", "Qwen/Qwen3.5-9B", "Qwen/Qwen3-4B-Instruct-2507", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-0.6B", "openbmb/MiniCPM5-2B"] {allow-input: true}
REVISION = ""  # @param {type:"string"}
WEIGHTS = "auto"  # @param ["auto", "16bit", "bnb-8bit", "bnb-4bit"]
MAX_SEQ_LENGTH = 4096  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}
AUTHORED144 = True  # @param {type:"boolean"}
PERTURBATIONS108 = True  # @param {type:"boolean"}
WANLI256 = False  # @param {type:"boolean"}
EXTRA_SUITE_JSONL = ""  # @param {type:"string"}
COMPARE_REFERENCE = True  # @param {type:"boolean"}

### GPU and precision

`WEIGHTS = "auto"` keeps 16-bit weights whenever they fit, because that preserves
the reference accuracy, and drops to 4-bit NF4 only when they do not. Choose it
yourself to trade accuracy for memory:

| Card | VRAM | Qwen3.5-4B suggestion |
| --- | --- | --- |
| A100 / L4 / RTX 3090-4090 | 22-80 GB | `16bit` in bf16, the reference setting |
| T4 / V100 / RTX 2080 Ti | 11-16 GB | `16bit` in fp16 (no bf16 before Ampere) |
| RTX 2060-3070, 8 GB | 6-8 GB | `bnb-4bit`, or a smaller base model |

`bnb-8bit` is the most faithful quantized format but is usually *slower* than
16-bit; `bnb-4bit` is smaller and faster than 8-bit. Unsloth's Triton kernels
need compute capability 7.0 or newer (V100, T4, RTX 20xx and later).

In [ ]:
import unsloth  # import before transformers so Unsloth can patch it
import torch
from huggingface_hub import model_info
from semif_phase1 import unsloth_backend as ub

GPU = ub.gpu_profile()
print(f"{GPU['name']}: {GPU['total_gib']} GiB, compute {GPU['compute_capability']}, bf16={GPU['bf16']}")
if float(GPU["compute_capability"]) < 7.0:
    print("WARNING: below compute 7.0; Unsloth's Triton kernels are unsupported on this card.")

revision = REVISION or MODELS.get(MODEL)
WEIGHTS_USED = WEIGHTS
if WEIGHTS == "auto":
    if Path(MODEL).exists():
        WEIGHTS_USED = "16bit"
    else:
        info = model_info(MODEL, revision=revision)
        params = info.safetensors.total if info.safetensors else None
        WEIGHTS_USED = ub.suggest_weights(params, GPU["total_bytes"]) if params else "16bit"
        print(f"{MODEL}: {params / 1e9:.2f}B parameters" if params else f"{MODEL}: parameter count unknown")
print(f"weights: {WEIGHTS_USED}")

### Load through Unsloth

Qwen3.5 checkpoints are vision-language models and load with `FastVisionModel`; text-only models use `FastLanguageModel`. The resolved commit, precision and GPU are recorded in every result row.

In [ ]:
model, tokenizer, META = ub.load_model(
    MODEL,
    revision,
    load_in_4bit = WEIGHTS_USED == "bnb-4bit",
    load_in_8bit = WEIGHTS_USED == "bnb-8bit",
    max_seq_length = MAX_SEQ_LENGTH,
)
META["semlf_commit"] = COMMIT
print(json.dumps(META, indent = 1))
print(f"{torch.cuda.memory_allocated() / 2**30:.2f} GiB allocated after loading")

### Suites

* `authored144`: the owned 144-row decision set behind the README's 0.813 balanced accuracy.
* `perturbations108`: output-blind rewrites of the same items (option order, wording).
* `wanli256`: the frozen WANLI test selection, fetched from its pinned Hub revision and SHA-256 checked.

Any extra labeled JSONL (rows with `label`, `family`, `group_id`) can be added as a suite.

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import colab_helpers as ch

names = [name for name, keep in (("authored144", AUTHORED144), ("perturbations108", PERTURBATIONS108),
                                 ("wanli256", WANLI256)) if keep]
if EXTRA_SUITE_JSONL:
    names.append(EXTRA_SUITE_JSONL)
SUITES = ch.load_suites(names, OUT, ROOT)
print({name: len(rows) for name, rows in SUITES.items()})

### Padding parity

Batching left-pads prompts. Before trusting batched numbers on a new model or precision, confirm they match single-row scoring.

In [ ]:
from semif_phase1.unsloth_backend import compare_predictions, score_rows

probe = next(iter(SUITES.values()))[:32]
single = score_rows(model, tokenizer, probe, META, mode = "direct")
batched = score_rows(model, tokenizer, probe, META, mode = "batched", batch_size = BATCH_SIZE)
parity = compare_predictions(single, batched)
print({key: value for key, value in parity.items() if key != "flips"})
if parity["argmax_agreement"] < 1:
    print("WARNING: batching changed decisions on this model; set BATCH_SIZE = 1 for exact parity.")

### Evaluate

`delta_95ci` is this run's balanced accuracy minus the reference's, with a paired source-group bootstrap interval; an interval that spans 0 means no detectable change. `argmax_agree` and `mean_tv` (total variation) measure how much individual decisions moved.

In [ ]:
def scorer(model, tokenizer, meta, batch_size):
    return lambda rows, progress: score_rows(model, tokenizer, rows, meta, mode = "batched",
                                             batch_size = batch_size, progress = progress)

REPORT, PREDICTIONS = ch.evaluate_suites(scorer(model, tokenizer, META, BATCH_SIZE), SUITES,
                                         COMPARE_REFERENCE, ROOT, tqdm)
pd.DataFrame(ch.summary_rows(REPORT, model = MODEL, weights = WEIGHTS_USED, batch = BATCH_SIZE))

In [ ]:
RUN_TAG = f"{MODEL.replace('/', '_')}-{WEIGHTS_USED}-{time.strftime('%Y%m%d-%H%M%S')}"
ch.write_create_only(OUT / f"eval-{RUN_TAG}.json", {"model": META, "gpu": GPU, "batch_size": BATCH_SIZE, "report": REPORT})
for name, rows in PREDICTIONS.items():
    ch.write_create_only(OUT / f"eval-{RUN_TAG}.{name}.predictions.jsonl", rows)
print(f"wrote {OUT}/eval-{RUN_TAG}.*")

Decisions that flipped relative to the reference, if any:

In [ ]:
pd.DataFrame([dict(suite = name, **flip) for name, entry in REPORT.items() for flip in entry.get("flips", [])])

### Sweep precisions, batch sizes and base models

Entries are `model | weights | batch`, separated by `;`. The model loaded above is released first, then every configuration is loaded, scored and freed in turn. `pick_config` returns the fastest configuration within `MAX_DROP` balanced accuracy of the best one; set `AGREEMENT_FLOOR` to also require argmax agreement with the Qwen3.5-4B reference (leave it at 0 when comparing different base models).

In [ ]:
RUN_SWEEP = False  # @param {type:"boolean"}
SWEEP = "Qwen/Qwen3.5-4B | 16bit | 16; Qwen/Qwen3.5-4B | bnb-8bit | 16; Qwen/Qwen3.5-4B | bnb-4bit | 16; Qwen/Qwen3.5-2B | 16bit | 16"  # @param {type:"string"}
SELECT_SUITE = "authored144"  # @param {type:"string"}
MAX_DROP = 0.02  # @param {type:"number"}
AGREEMENT_FLOOR = 0.0  # @param {type:"number"}

SWEEP_ROWS = []
if RUN_SWEEP:
    del model, tokenizer
    ch.release_cuda()
    for config in ch.parse_sweep(SWEEP, MODELS):
        print(f"--- {config}")
        model, tokenizer, meta = ub.load_model(
            config["model"], config["revision"], max_seq_length = MAX_SEQ_LENGTH,
            load_in_4bit = config["weights"] == "bnb-4bit", load_in_8bit = config["weights"] == "bnb-8bit")
        report, _ = ch.evaluate_suites(scorer(model, tokenizer, meta, config["batch_size"]), SUITES,
                                       COMPARE_REFERENCE, ROOT, tqdm)
        SWEEP_ROWS += ch.summary_rows(report, model = config["model"], weights = config["weights"],
                                      batch = config["batch_size"])
        del model, tokenizer
        ch.release_cuda()
    ch.write_create_only(OUT / f"sweep-{time.strftime('%Y%m%d-%H%M%S')}.json", {"gpu": GPU, "rows": SWEEP_ROWS})
    from IPython.display import display
    display(pd.DataFrame(SWEEP_ROWS))
    print("pick:", ch.pick_config(SWEEP_ROWS, SELECT_SUITE, MAX_DROP, AGREEMENT_FLOOR or None))

### Notes

* Probabilities are conditional option scores over the supplied options, not
  calibrated confidences. Validate them on your own workload before acting on them.
* Outputs are create-only, matching the repository's benchmark policy: every run
  writes a new file under `outputs/`.
* The Unsloth install cell follows
  [unslothai/notebooks](https://github.com/unslothai/notebooks) (LGPL-3.0).